In [1]:
import plotly.graph_objects as go
import networkx as nx
import pandas as pd

In [2]:
DATA_PATH = "../data/processed/final_gene_features.csv"
EDGE_PATH = "../data/processed/final_edge_list.csv"

df = pd.read_csv(DATA_PATH)
edges = pd.read_csv(EDGE_PATH)

Build Network Graph

In [3]:
G = nx.from_pandas_edgelist(
    edges,
    source="gene1",
    target="gene2"
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 16185
Edges: 236000


Generate 3-D Layout

NetworkX computes coordinates for each node.

This places connected genes closer together in 3-D space.

In [4]:
pos = nx.spring_layout(
    G,
    dim=3,
    seed=42
)

Extract Node Coordinates

In [5]:
x_nodes = []
y_nodes = []
z_nodes = []

for node in G.nodes():
    x_nodes.append(pos[node][0])
    y_nodes.append(pos[node][1])
    z_nodes.append(pos[node][2])

Extract Edge Coordinates

In [6]:
x_edges = []
y_edges = []
z_edges = []

for edge in G.edges():
    x0, y0, z0 = pos[edge[0]]
    x1, y1, z1 = pos[edge[1]]

    x_edges += [x0, x1, None]
    y_edges += [y0, y1, None]
    z_edges += [z0, z1, None]

Node Colors by Label

Pathogenic genes will appear differently.

In [7]:
gene_labels = dict(
    zip(df["GeneSymbol"], df["label"])
)

node_colors = [
    gene_labels.get(node,0)
    for node in G.nodes()
]

Create Edge Trace

In [8]:
edge_trace = go.Scatter3d(
    x=x_edges,
    y=y_edges,
    z=z_edges,
    mode="lines",
    line=dict(color="lightgray", width=1),
    hoverinfo="none"
)

Create Node Trace

In [9]:
node_trace = go.Scatter3d(
    x=x_nodes,
    y=y_nodes,
    z=z_nodes,
    mode="markers",
    marker=dict(
        size=4,
        color=node_colors,
        colorscale="Viridis",
        opacity=0.8
    ),
    text=list(G.nodes()),
    hoverinfo="text"
)

Create Interactive Figure

In [13]:
import plotly.io as pio
pio.renderers.default = "browser"

In [16]:
fig.show()

In [15]:
fig = go.Figure(
    data=[edge_trace, node_trace]
)

fig.update_layout(
    title="3D Gene Interaction Network",
    showlegend=False,
    margin=dict(l=0,r=0,b=0,t=40)
)

fig.show()

Interactive controls allow you to:

Rotate the network

Zoom in/out

Hover over genes

Explore clusters

📊 What This Visualization Shows

The 3-D graph reveals:

• hub genes (high connectivity)
• network clusters (biological modules)
• pathogenic gene regions

Genes that cluster together often share biological functions or pathways.